# Запуск пар DLightSB и DLightSB-MM в Colab

Ноутбук запускает пары экспериментов для размерностей 2 16 и 64

Для каждой размерности предусмотрены три отдельные ячейки

1. Запуск baseline DLightSB
2. Запуск DLightSB-MM
3. Сбор всех результатов в один ZIP-архив и скачивание

Запускайте только секции нужных размерностей


## Подготовка окружения

В Colab выберите GPU runtime затем выполните эту ячейку один раз


In [ ]:
from pathlib import Path
import os
import subprocess

REPOSITORY_URL = "https://github.com/fedorkolotilin3/catsbench_mm.git"
BRANCH = "master"
PROJECT_DIR = Path("/content/catsbench_mm")

if not (PROJECT_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            REPOSITORY_URL,
            str(PROJECT_DIR),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "fetch", "origin"],
        cwd=PROJECT_DIR,
        check=True,
    )
    subprocess.run(
        ["git", "switch", BRANCH],
        cwd=PROJECT_DIR,
        check=True,
    )
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=PROJECT_DIR,
        check=True,
    )

subprocess.run(
    ["bash", "scripts/setup_colab.sh"],
    cwd=PROJECT_DIR,
    check=True,
)

os.chdir(PROJECT_DIR)
print("Проект готов:", PROJECT_DIR)


## Функции запуска

Эта ячейка не запускает обучение


In [ ]:
from datetime import datetime, timezone
from time import perf_counter
import csv
import hashlib
import json
import zipfile

SEED = 42
EXPORT_DIR = Path("/content/catsbench_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PAIR_RUNS = {}


In [ ]:
import codecs
import sys


def run_streaming(command: list[str], cwd: Path) -> None:
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )

    if process.stdout is None:
        raise RuntimeError("Не удалось подключиться к выводу процесса")

    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")

    try:
        while True:
            chunk = process.stdout.read(4096)

            if not chunk:
                break

            sys.stdout.write(decoder.decode(chunk))
            sys.stdout.flush()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()

    tail = decoder.decode(b"", final=True)

    if tail:
        sys.stdout.write(tail)
        sys.stdout.flush()

    return_code = process.wait()

    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


In [ ]:
def experiment_name(kind: str, dim: int) -> str:
    if kind == "baseline":
        return f"dlight_sb/benchmark_hd/d{dim}_g002"
    if kind == "mm":
        return f"dlight_sb_mm/benchmark_hd/d{dim}_g002"
    raise ValueError(f"Неизвестный тип эксперимента: {kind}")


def latest_completed_run(experiment: str) -> Path:
    parent = PROJECT_DIR / "logs" / "runs" / Path(experiment) / str(SEED)

    if not parent.is_dir():
        raise FileNotFoundError(f"Каталог эксперимента не найден: {parent}")

    candidates = [
        path
        for path in parent.iterdir()
        if path.is_dir()
        and (path / "checkpoints" / "last.ckpt").is_file()
    ]

    if not candidates:
        raise FileNotFoundError(
            f"Нет завершённого запуска с last.ckpt в {parent}"
        )

    return max(candidates, key=lambda path: path.name)


def run_experiment(kind: str, dim: int) -> Path:
    experiment = experiment_name(kind, dim)

    if kind == "baseline":
        command = [
            "bash",
            "scripts/run_baseline_train_test.sh",
            "--experiment",
            experiment,
        ]
    else:
        command = [
            "bash",
            "scripts/run_mm_train_test.sh",
            "--device",
            "gpu",
            "--experiment",
            experiment,
        ]

    print(f"Запуск {kind} для d={dim}", flush=True)
    started_at = perf_counter()
    run_streaming(command, PROJECT_DIR)
    elapsed_seconds = perf_counter() - started_at

    run_dir = latest_completed_run(experiment)
    PAIR_RUNS.setdefault(dim, {})[kind] = run_dir

    print(f"Завершено за {elapsed_seconds / 60:.1f} мин: {run_dir}")
    return run_dir


## Функции проверки и экспорта

Архив содержит полные каталоги запусков baseline и MM включая CSV логи validation и test Hydra конфиги текстовые логи и checkpoints


In [ ]:
def sha256sum(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def find_metrics_csv(run_dir: Path, stage: str) -> Path:
    matches = sorted(run_dir.glob(f"*/{stage}/metrics.csv"))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Ожидался один {stage}/metrics.csv в {run_dir} найдено {len(matches)}"
        )

    return matches[0]


def validate_run(run_dir: Path) -> dict:
    checkpoint = run_dir / "checkpoints" / "last.ckpt"
    train_metrics = find_metrics_csv(run_dir, "train")
    test_metrics = find_metrics_csv(run_dir, "test")

    required = [
        checkpoint,
        train_metrics,
        test_metrics,
        run_dir / ".hydra" / "config.yaml",
        run_dir / ".hydra" / "overrides.yaml",
        run_dir / "train.log",
        run_dir / "test.log",
    ]

    missing = [str(path) for path in required if not path.is_file()]

    if missing:
        raise FileNotFoundError(
            "В запуске отсутствуют обязательные файлы:\n"
            + "\n".join(missing)
        )

    with train_metrics.open(encoding="utf-8", newline="") as file:
        columns = next(csv.reader(file))

    validation_columns = [
        column
        for column in columns
        if column.startswith("val/")
    ]

    if not validation_columns:
        raise RuntimeError(
            f"В {train_metrics} нет validation-метрик"
        )

    return {
        "run_dir": str(run_dir),
        "last_checkpoint": str(checkpoint.relative_to(run_dir)),
        "last_checkpoint_bytes": checkpoint.stat().st_size,
        "last_checkpoint_sha256": sha256sum(checkpoint),
        "train_metrics": str(train_metrics.relative_to(run_dir)),
        "test_metrics": str(test_metrics.relative_to(run_dir)),
        "validation_columns": validation_columns,
    }


def add_directory_to_zip(archive: zipfile.ZipFile, source: Path, prefix: str):
    for path in sorted(source.rglob("*")):
        if path.is_file():
            archive.write(
                path,
                arcname=str(Path(prefix) / path.relative_to(source)),
            )


def export_pair(dim: int, download: bool = True) -> Path:
    runs = {}

    for kind in ("baseline", "mm"):
        run_dir = PAIR_RUNS.get(dim, {}).get(kind)

        if run_dir is None or not Path(run_dir).is_dir():
            run_dir = latest_completed_run(experiment_name(kind, dim))

        runs[kind] = Path(run_dir)

    run_info = {
        kind: validate_run(run_dir)
        for kind, run_dir in runs.items()
    }

    commit = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=PROJECT_DIR,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()

    created_at = datetime.now(timezone.utc)
    timestamp = created_at.strftime("%Y-%m-%d_%H-%M-%S_UTC")
    archive_path = EXPORT_DIR / f"catsbench_d{dim}_g002_pair_{timestamp}.zip"

    manifest = {
        "created_at_utc": created_at.isoformat(),
        "repository": REPOSITORY_URL,
        "branch": BRANCH,
        "commit": commit,
        "seed": SEED,
        "dimension": dim,
        "experiments": {
            kind: {
                "hydra_experiment": experiment_name(kind, dim),
                **run_info[kind],
            }
            for kind in ("baseline", "mm")
        },
    }

    with zipfile.ZipFile(
        archive_path,
        mode="w",
        compression=zipfile.ZIP_STORED,
        allowZip64=True,
    ) as archive:
        archive.writestr(
            "manifest.json",
            json.dumps(manifest, ensure_ascii=False, indent=2),
        )
        add_directory_to_zip(archive, runs["baseline"], "baseline")
        add_directory_to_zip(archive, runs["mm"], "mm")

    print("Архив готов:", archive_path)
    print(f"Размер: {archive_path.stat().st_size / 1024**2:.1f} MiB")
    print("Финальные checkpoint находятся в baseline/checkpoints/last.ckpt и mm/checkpoints/last.ckpt")

    if download:
        try:
            from google.colab import files
            files.download(str(archive_path))
        except ImportError:
            print("Автоматическое скачивание доступно только в Google Colab")

    return archive_path


# Пара экспериментов для размерности 2


## Baseline DLightSB d2


In [ ]:
run_experiment("baseline", 2)


## DLightSB-MM d2


In [ ]:
run_experiment("mm", 2)


## Скачать один архив с парой d2


In [ ]:
export_pair(2)


# Пара экспериментов для размерности 16


## Baseline DLightSB d16


In [ ]:
run_experiment("baseline", 16)


## DLightSB-MM d16


In [ ]:
run_experiment("mm", 16)


## Скачать один архив с парой d16


In [ ]:
export_pair(16)


# Пара экспериментов для размерности 64


## Baseline DLightSB d64


In [ ]:
run_experiment("baseline", 64)


## DLightSB-MM d64


In [ ]:
run_experiment("mm", 64)


## Скачать один архив с парой d64


In [ ]:
export_pair(64)


## Содержимое архива

Каждый ZIP содержит

- `manifest.json` с commit hash параметрами запуска путями и контрольными суммами checkpoint
- `baseline` с полным каталогом baseline-запуска
- `mm` с полным каталогом MM-запуска
- `csv/train/metrics.csv` с историей train и validation
- `csv/test/metrics.csv` с итоговыми метриками
- `.hydra` с полной конфигурацией
- `train.log` и `test.log`
- `checkpoints/last.ckpt` для продолжения обучения повторной validation или test
